In [18]:
# pip install QuantLib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import QuantLib as ql

# Part 1A: LIBOR legacy curve

In [ ]:
def parse_tenor_to_period(tenor: str) -> ql.Period:
    """
    Convert strings like '6m', '1y', '2w', '10d' into QuantLib Periods.
    """
    t = tenor.strip().lower().replace(" ", "")
    if t.endswith("d"):
        return ql.Period(int(t[:-1]), ql.Days)
    if t.endswith("w"):
        return ql.Period(int(t[:-1]), ql.Weeks)
    if t.endswith("m"):
        return ql.Period(int(t[:-1]), ql.Months)
    if t.endswith("y"):
        return ql.Period(int(t[:-1]), ql.Years)
    raise ValueError(f"Unsupported tenor format: {tenor}")


def ql_date_from_string(date_str: str) -> ql.Date:
    """
    Convert 'YYYY-MM-DD' string into QuantLib Date.
    """
    ts = pd.Timestamp(date_str)
    return ql.Date(ts.day, ts.month, ts.year)


def set_evaluation_date(eval_date: str = "2026-01-01") -> ql.Date:
    """
    Set QuantLib global evaluation date.
    """
    qld = ql_date_from_string(eval_date)
    ql.Settings.instance().evaluationDate = qld
    return qld


def ensure_decimal_rates(df: pd.DataFrame, rate_col: str = "Rate") -> pd.DataFrame:
    """
    Convert percentage rates into decimals if needed.
    Example: 4.50 -> 0.045
    """
    out = df.copy()
    if out[rate_col].abs().max() > 1:
        out[rate_col] = out[rate_col] / 100.0
    return out


def tenor_to_years(tenor: str) -> float:
    """
    Rough conversion for plotting/reporting.
    """
    t = tenor.strip().lower().replace(" ", "")
    if t.endswith("d"):
        return int(t[:-1]) / 365.0
    if t.endswith("w"):
        return int(t[:-1]) * 7 / 365.0
    if t.endswith("m"):
        return int(t[:-1]) / 12.0
    if t.endswith("y"):
        return float(int(t[:-1]))
    raise ValueError(f"Unsupported tenor format: {tenor}")


def make_ql_calendar(name: str = "US_GOVT") -> ql.Calendar:
    """
    Centralized calendar builder so later parts remain consistent.
    """
    if name == "US_GOVT":
        return ql.UnitedStates(ql.UnitedStates.GovernmentBond)
    if name == "TARGET":
        return ql.TARGET()
    return ql.NullCalendar()


def extract_curve_points(
    curve_handle: ql.YieldTermStructureHandle,
    tenors: list[str],
    calendar: ql.Calendar,
    bdc=ql.ModifiedFollowing,
    zero_day_count=ql.Actual365Fixed(),
    zero_compounding=ql.Continuous
) -> pd.DataFrame:
    """
    Extract discount factors and zero rates from a curve at selected tenors.
    """
    curve = curve_handle.currentLink()
    ref_date = curve.referenceDate()

    rows = []
    for tenor in tenors:
        maturity = calendar.advance(ref_date, parse_tenor_to_period(tenor), bdc)
        df = curve.discount(maturity)
        zr = curve.zeroRate(maturity, zero_day_count, zero_compounding).rate()

        rows.append({
            "Tenor": tenor,
            "Years": tenor_to_years(tenor),
            "MaturityDate": pd.Timestamp(maturity.year(), maturity.month(), maturity.dayOfMonth()),
            "DiscountFactor": df,
            "ZeroRate_Cont(%)": 100 * zr
        })

    return pd.DataFrame(rows)


def plot_discount_curve(
    curve_df: pd.DataFrame,
    title: str,
    x_col: str = "Years",
    y_col: str = "DiscountFactor"
):
    """
    Simple reusable plotting function for discount curves.
    """
    plt.figure(figsize=(9, 5))
    plt.plot(curve_df[x_col], curve_df[y_col], marker="o")
    plt.title(title)
    plt.xlabel("Maturity (Years)")
    plt.ylabel("Discount Factor")
    plt.grid(True, alpha=0.3)
    plt.show()


def compare_curve_tables(
    left_df: pd.DataFrame,
    right_df: pd.DataFrame,
    left_name: str = "LeftCurve",
    right_name: str = "RightCurve"
) -> pd.DataFrame:
    """
    Reusable curve comparison table. Useful later for single vs multi-curve comparison.
    """
    merged = left_df.merge(
        right_df[["Tenor", "DiscountFactor", "ZeroRate_Cont(%)"]],
        on="Tenor",
        suffixes=(f"_{left_name}", f"_{right_name}")
    )
    merged[f"DF_Diff_{left_name}-{right_name}"] = (
        merged[f"DiscountFactor_{left_name}"] - merged[f"DiscountFactor_{right_name}"]
    )
    merged[f"ZeroRateDiff_bp_{left_name}-{right_name}"] = 100 * (
        merged[f"ZeroRate_Cont(%)_{left_name}"] - merged[f"ZeroRate_Cont(%)_{right_name}"]
    )
    return merged

In [ ]:
def build_piecewise_discount_curve(
    helpers: list,
    eval_date: str = "2026-01-01",
    day_count=ql.Actual365Fixed(),
    interpolation: str = "linear_discount"
) -> ql.YieldTermStructureHandle:
    """
    Build a piecewise discount curve from QuantLib helpers.

    interpolation choices:
    - 'linear_discount'      -> preferred if your QuantLib build supports PiecewiseLinearDiscount
    - 'loglinear_discount'   -> safe fallback in some QuantLib builds
    """
    eval_qld = set_evaluation_date(eval_date)

    if interpolation == "linear_discount":
        if hasattr(ql, "PiecewiseLinearDiscount"):
            curve = ql.PiecewiseLinearDiscount(eval_qld, helpers, day_count)
        else:
            raise AttributeError(
                "Your QuantLib build does not expose PiecewiseLinearDiscount. "
                "Use interpolation='loglinear_discount' as a fallback, "
                "or install a build that supports PiecewiseLinearDiscount."
            )
    elif interpolation == "loglinear_discount":
        curve = ql.PiecewiseLogLinearDiscount(eval_qld, helpers, day_count)
    else:
        raise ValueError("Unsupported interpolation choice.")

    curve.enableExtrapolation()
    return ql.YieldTermStructureHandle(curve)

In [ ]:
def make_libor_legacy_single_curve_helpers(
    market_df: pd.DataFrame,
    settlement_days: int = 2,
    calendar: ql.Calendar = None,
    deposit_day_count=ql.Actual360(),
    fixed_leg_frequency=ql.Semiannual,
    fixed_leg_convention=ql.ModifiedFollowing,
    fixed_leg_day_count=None
) -> list:
    """
    Build QuantLib helpers for the legacy LIBOR single-curve framework:
    - short end from LIBOR deposit
    - longer end from IRS swaps
    """
    if calendar is None:
        calendar = make_ql_calendar("US_GOVT")

    if fixed_leg_day_count is None:
        fixed_leg_day_count = ql.Thirty360(ql.Thirty360.BondBasis)

    df = ensure_decimal_rates(market_df, "Rate")

    helpers = []

    # 6M LIBOR used as floating index for swaps in legacy single-curve framework
    libor_6m = ql.USDLibor(ql.Period(6, ql.Months))

    # Deposit helpers
    depo = df[df["Product"].str.upper().str.contains("LIBOR", na=False)].copy()
    for _, row in depo.iterrows():
        helpers.append(
            ql.DepositRateHelper(
                ql.QuoteHandle(ql.SimpleQuote(float(row["Rate"]))),
                parse_tenor_to_period(row["Tenor"]),
                settlement_days,
                calendar,
                ql.ModifiedFollowing,
                False,
                deposit_day_count
            )
        )

    # Swap helpers
    swaps = df[df["Product"].str.upper().str.contains("IRS", na=False)].copy()
    for _, row in swaps.iterrows():
        helpers.append(
            ql.SwapRateHelper(
                ql.QuoteHandle(ql.SimpleQuote(float(row["Rate"]))),
                parse_tenor_to_period(row["Tenor"]),
                calendar,
                fixed_leg_frequency,
                fixed_leg_convention,
                fixed_leg_day_count,
                libor_6m
            )
        )

    return helpers

In [ ]:
def build_libor_legacy_single_curve(
    market_df: pd.DataFrame,
    eval_date: str = "2026-01-01",
    settlement_days: int = 2,
    calendar: ql.Calendar = None,
    interpolation: str = "linear_discount"
) -> ql.YieldTermStructureHandle:
    """
    Build the Part 1A LIBOR legacy single-curve discount curve.
    """
    if calendar is None:
        calendar = make_ql_calendar("US_GOVT")

    helpers = make_libor_legacy_single_curve_helpers(
        market_df=market_df,
        settlement_days=settlement_days,
        calendar=calendar
    )

    curve_handle = build_piecewise_discount_curve(
        helpers=helpers,
        eval_date=eval_date,
        day_count=ql.Actual365Fixed(),
        interpolation=interpolation
    )

    return curve_handle

In [ ]:
libor_legacy_df = pd.DataFrame({
    "Tenor":   ["6m", "1y", "2y", "3y", "4y", "5y", "7y", "10y", "15y", "20y", "30y"],
    "Product": ["LIBOR", "IRS", "IRS", "IRS", "IRS", "IRS", "IRS", "IRS", "IRS", "IRS", "IRS"],
    "Rate":    [4.50, 4.80, 5.00, 5.15, 5.25, 5.30, 5.50, 5.70, 6.00, 6.50, 7.00]
})

libor_legacy_df

In [ ]:
us_cal = make_ql_calendar("US_GOVT")

libor_single_curve = build_libor_legacy_single_curve(
    market_df=libor_legacy_df,
    eval_date="2026-01-01",
    settlement_days=2,
    calendar=us_cal,
    interpolation="linear_discount"   # fallback to "loglinear_discount" if needed
)

In [ ]:
tenors_part1a = ["6m", "1y", "2y", "3y", "4y", "5y", "7y", "10y", "15y", "20y", "30y"]

libor_single_curve_df = extract_curve_points(
    curve_handle=libor_single_curve,
    tenors=tenors_part1a,
    calendar=us_cal,
    bdc=ql.ModifiedFollowing
)

libor_single_curve_df

In [ ]:
plot_discount_curve(
    libor_single_curve_df,
    title="Part 1A — Legacy LIBOR Single-Curve Discount Factors"
)